# Tutorial 1: FieldCoords

`FieldCoords` is the foundational datatype in StarSharp.  It represents field
coordinates — positions on the sky or on the focal plane — in any of several
coordinate **frames** and **spaces**.

## Key concepts

| Concept | Options | Description |
|---------|---------|-------------|
| **Frame** | `ocs`, `ccs`, `dvcs`, `edcs` | Orientation of the coordinate axes |
| **Space** | `angle`, `focal_plane` | Physical units (angular vs. length) |

**Frames:**
- **OCS** (Optical Coordinate System) — aligned with the optics; the natural frame for ray tracing.  See [SITCOMTN-003](https://sitcomtn-003.lsst.io/#the-optical-coordinate-system).
- **CCS** (Camera Coordinate System) — rotated from OCS by the rotator position angle (`rtp`).
- **EDCS** (Engineering Diagram CS) — synonym for CCS (same orientation, different label).  See [LSE-349](ls.st/LSE-349).
- **DVCS** (Data Visualization CS) — transpose (x↔y swap) of CCS=DVCS.

*Warning* Currently StarSharp DVCS is only consistent with data management DVCS for the science sensors.

**Spaces:**
- **angle** — field‐angle coordinates (e.g. degrees, arcsec).
- **focal_plane** — physical positions on the focal plane (e.g. mm, µm). Requires a WCS.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import Angle
import galsim

from StarSharp import FieldCoords

%matplotlib inline

## 1. Construction

`FieldCoords` is a frozen dataclass.  The minimal construction requires `x` and
`y` as `astropy.units.Quantity` arrays.  The unit type determines the **space**
automatically: angular units → `'angle'`, length units → `'focal_plane'`.

In [ ]:
# Angular field coordinates (degrees)
fc_angle = FieldCoords(
    x=np.array([0.0, 0.5, -0.7, 1.2]) * u.deg,
    y=np.array([0.3, -0.4, 0.8, -0.1]) * u.deg,
)
print(f"Space: {fc_angle.space}")
print(f"Frame: {fc_angle.frame}")
print(f"nfield: {fc_angle.nfield}")
fc_angle

In [ ]:
# Arcseconds work too — still angular space
fc_arcsec = FieldCoords(
    x=np.array([100.0, -200.0]) * u.arcsec,
    y=np.array([50.0, 150.0]) * u.arcsec,
)
print(f"Space: {fc_arcsec.space}")

In [ ]:
# Focal-plane coordinates (millimeters)
fc_fp = FieldCoords(
    x=np.array([10.0, -30.0, 50.0]) * u.mm,
    y=np.array([-20.0, 40.0, -15.0]) * u.mm,
    frame="ccs",
)
print(f"Space: {fc_fp.space}")
print(f"Frame: {fc_fp.frame}")

In [ ]:
# Scalar coordinates are automatically promoted to 1-D arrays
fc_scalar = FieldCoords(x=0.5 * u.deg, y=-0.3 * u.deg)
print(f"nfield: {fc_scalar.nfield}")
print(f"x shape: {fc_scalar.x.shape}")

## 2. Frame conversions

Frame conversions are available as properties: `.ocs`, `.ccs`, `.dvcs`, `.edcs`.

Converting between OCS and CCS requires the **rotator position angle** (`rtp`).
CCS ↔ EDCS is a simple relabel; CCS ↔ DVCS swaps x and y.

In [ ]:
# Define a rotator position angle
rtp = Angle(30, unit=u.deg)

# Create field coords with rtp so frame conversions work
fc = FieldCoords(
    x=np.array([1.0, 0.0, -0.5, 0.8]) * u.deg,
    y=np.array([0.0, 1.0, 0.7, -0.3]) * u.deg,
    frame="ocs",
    rtp=rtp,
)
print(f"Original frame: {fc.frame}")
print(f"  x = {fc.x}")
print(f"  y = {fc.y}")

In [ ]:
# OCS → CCS: rotation by rtp
fc_ccs = fc.ccs
print(f"CCS frame: {fc_ccs.frame}")
print(f"  x = {fc_ccs.x}")
print(f"  y = {fc_ccs.y}")

In [ ]:
# CCS → EDCS: just a relabel (same values)
fc_edcs = fc_ccs.edcs
print(f"EDCS frame: {fc_edcs.frame}")
print(f"  x matches CCS: {np.allclose(fc_ccs.x, fc_edcs.x)}")

In [ ]:
# CCS → DVCS: x↔y swap
fc_dvcs = fc_ccs.dvcs
print(f"DVCS frame: {fc_dvcs.frame}")
print(f"  DVCS.x == CCS.y: {np.allclose(fc_dvcs.x, fc_ccs.y)}")
print(f"  DVCS.y == CCS.x: {np.allclose(fc_dvcs.y, fc_ccs.x)}")

In [ ]:
# Roundtrip: OCS → CCS → OCS should recover the original
fc_roundtrip = fc.ccs.ocs
print(f"Roundtrip x close: {np.allclose(fc.x, fc_roundtrip.x)}")
print(f"Roundtrip y close: {np.allclose(fc.y, fc_roundtrip.y)}")
print(f"Max difference: {np.max(np.abs(fc.x - fc_roundtrip.x)):.2e}")

### Visualizing frame conversions

In [ ]:
xs = np.linspace(-317, 317, 15) * u.mm  # Approximate geometry of E2V sensors
xs, ys = np.meshgrid(xs, xs)
w = (ys > 200 * u.mm)
w |= (ys < -200 * u.mm)
w |= (xs < -200 * u.mm) & (ys < 75 * u.mm)
# Remove

fc_grid = FieldCoords(x=xs[~w].flatten(), y=ys[~w].flatten(), frame="ccs", rtp=rtp)
fig, axes = plt.subplots(1, 4, figsize=(16, 4), constrained_layout=True)

for ax, (label, fld) in zip(axes, [
    ("OCS", fc_grid.ocs),
    ("CCS", fc_grid.ccs),
    ("EDCS", fc_grid.edcs),
    ("DVCS", fc_grid.dvcs),
]):
    ax.scatter(fld.x.to_value(u.mm), fld.y.to_value(u.mm), s=80, zorder=5)
    ax.set_xlabel(f"{label} x [mm]")
    ax.set_ylabel(f"{label} y [mm]")
    ax.set_title(label)
    ax.set_aspect("equal")
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-400, 400)
    ax.set_ylim(-400, 400)

fig.suptitle(f"Field coordinates in all four frames (rtp = {rtp.deg:.0f}°)", fontsize=13)
plt.show()

## 3. Space conversions  (angle ↔ focal_plane)

Converting between angular and focal-plane space requires a **WCS** object.
The WCS maps OCS field angles (radians) ↔ CCS focal-plane positions (mm).

For this tutorial we build a simple affine WCS with a known plate scale.
In practice, `FieldCoords.from_builder()` computes a precise WCS from a
`batoid_rubin.LSSTBuilder`.

In [ ]:
# Build a simple affine WCS: 20 arcsec/mm plate scale, rotated by rtp
def make_wcs(rtp):
    c = float(np.cos(rtp.rad))
    s = float(np.sin(rtp.rad))
    scale = np.deg2rad(20 / 3600)  # 20 arcsec/mm in rad/mm
    rot = np.array([[c, -s], [s, c]]) * scale
    return galsim.AffineTransform(
        rot[0, 0], rot[0, 1], rot[1, 0], rot[1, 1],
        galsim.PositionD(0.0, 0.0),
    )

wcs = make_wcs(rtp)
print(type(wcs))

In [ ]:
# Attach the WCS to our FieldCoords
fc_with_wcs = FieldCoords(
    x=np.array([0.5, -0.3, 1.0]) * u.deg,
    y=np.array([0.2, 0.8, -0.6]) * u.deg,
    frame="ocs",
    rtp=rtp,
    wcs=wcs,
)
print(f"Space: {fc_with_wcs.space}")

In [ ]:
# Convert angle → focal_plane
fc_focal = fc_with_wcs.focal_plane
print(f"Space: {fc_focal.space}")
print(f"Frame: {fc_focal.frame}")
print(f"x = {fc_focal.x}")
print(f"y = {fc_focal.y}")

In [ ]:
# Convert back: focal_plane → angle
fc_back = fc_focal.angle
print(f"Space: {fc_back.space}")
print(f"Roundtrip x close: {np.allclose(fc_with_wcs.x, fc_back.x, atol=1e-10)}")
print(f"Max error: {np.max(np.abs(fc_with_wcs.x - fc_back.x)):.2e}")

In [ ]:
# Note: space conversion preserves the current frame
fc_ccs_angle = fc_with_wcs.ccs
fc_ccs_fp = fc_ccs_angle.focal_plane
print(f"Started in: frame={fc_ccs_angle.frame}, space={fc_ccs_angle.space}")
print(f"Ended in:   frame={fc_ccs_fp.frame}, space={fc_ccs_fp.space}")

### Visualizing space conversions

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

ax1.scatter(
    fc_with_wcs.x.to_value(u.deg),
    fc_with_wcs.y.to_value(u.deg),
    s=80, c=["C0", "C1", "C2"],
)
for i in range(fc_with_wcs.nfield):
    ax1.annotate(str(i), (fc_with_wcs.x[i].to_value(u.deg), fc_with_wcs.y[i].to_value(u.deg)),
                fontsize=10, ha='left', va='bottom')
ax1.set_xlabel("x [deg]")
ax1.set_ylabel("y [deg]")
ax1.set_title("Field-angle space (OCS)")
ax1.set_aspect("equal")
ax1.grid(True, alpha=0.3)

fp = fc_with_wcs.focal_plane
ax2.scatter(
    fp.x.to_value(u.mm),
    fp.y.to_value(u.mm),
    s=80, c=["C0", "C1", "C2"],
)
for i in range(fp.nfield):
    ax2.annotate(str(i), (fp.x[i].to_value(u.mm), fp.y[i].to_value(u.mm)),
                fontsize=10, ha='left', va='bottom')
ax2.set_xlabel("x [mm]")
ax2.set_ylabel("y [mm]")
ax2.set_title(f"Focal-plane space ({fp.frame.upper()})")
ax2.set_aspect("equal")
ax2.grid(True, alpha=0.3)

fig.suptitle("Angle ↔ Focal Plane space conversion", fontsize=13)

## 4. Indexing and batch dimensions

`FieldCoords` supports integer and slice indexing.  The last axis is the field
dimension; any leading axes are batch dimensions.

In [ ]:
# Integer indexing
fc0 = fc_with_wcs[0]
print(f"Single point: nfield={fc0.nfield}, x={fc0.x}")

# Slice indexing
fc_slice = fc_with_wcs[:2]
print(f"Sliced: nfield={fc_slice.nfield}")

In [ ]:
# Batch dimensions: 2D arrays have shape (batch, nfield)
rng = np.random.default_rng(42)
fc_batch = FieldCoords(
    x=rng.uniform(-1, 1, (3, 5)) * u.deg,
    y=rng.uniform(-1, 1, (3, 5)) * u.deg,
)
print(f"batch_shape: {fc_batch.batch_shape}")
print(f"nfield: {fc_batch.nfield}")
print(f"Total shape: {fc_batch.x.shape}")

## 5. Summary

| Property / Method | Description |
|---|---|
| `.frame` | Current frame: `'ocs'`, `'ccs'`, `'dvcs'`, `'edcs'` |
| `.space` | Current space: `'angle'` or `'focal_plane'` |
| `.ocs`, `.ccs`, `.dvcs`, `.edcs` | Frame conversion properties |
| `.angle`, `.focal_plane` | Space conversion properties (requires `wcs`) |
| `.nfield` | Number of field points |
| `.batch_shape` | Leading batch dimensions |
| `[idx]` | Indexing into field/batch dimensions |
| `.rtp` | Rotator position angle (needed for OCS↔CCS) |
| `.wcs` | WCS (needed for angle↔focal_plane) |

`FieldCoords` objects are carried by every other StarSharp datatype — they're
the common thread.  **Next:** [02_Spots.ipynb](02_Spots.ipynb) shows how spot
diagrams build on `FieldCoords`.